# R1 · ResearchLens results — every model, every graph, every table
CPU only (Kaggle or Colab). Downloads all saved results from your private Hugging Face dataset `researchlens-results`
and rebuilds the report figures: summary table, loss curves, training-vs-validation (over-fitting) curves,
train / val / test bars, cross-domain before/after, relation types, evidence ranking vs word overlap.
Needs the `HF_TOKEN` secret (Kaggle: Add-ons → Secrets; Colab: 🔑 Secrets).

In [ ]:
import os, sys
REPO_URL = "https://github.com/mkavinvelavan/ResearchLens.git"
ROOT = "/kaggle/working" if os.path.exists("/kaggle") else "/content"
if not os.path.exists(f"{ROOT}/researchlens_repo"):
    !git clone -q {REPO_URL} {ROOT}/researchlens_repo
else:
    !git -C {ROOT}/researchlens_repo pull -q
%cd {ROOT}/researchlens_repo
try:
    from kaggle_secrets import UserSecretsClient; os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    try:
        from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception as e:
        print("HF_TOKEN not found:", e)
from huggingface_hub import HfApi, snapshot_download
user = HfApi().whoami()["name"]
local = snapshot_download(f"{user}/researchlens-results", repo_type="dataset", local_dir=f"{ROOT}/rl_results")
RES = f"{local}/results"; FIG = f"{ROOT}/rl_figures"; os.makedirs(FIG, exist_ok=True)
!ls {RES}

## 1. All models at a glance (test scores, train/val/test, over-fitting gap)

In [ ]:
import pandas as pd
from IPython.display import display, Image
from researchlens.evaluation import summary as S
runs = S.discover(RES)
table = S.pct(S.summary(runs))
table.to_csv(f"{FIG}/all_models_summary.csv", index=False)
pd.set_option("display.max_colwidth", 60)
display(table)

## 2. Training curves for every model (loss · over-fitting check · train/val/test)

In [ ]:
made = S.curves(runs, FIG)
for p in made:
    display(Image(filename=str(p), width=520))

## 3. Does it work outside one field? (cross-domain, before vs after multi-domain training)

In [ ]:
cd = S.crossdomain(runs); display(cd)
cd.to_csv(f"{FIG}/crossdomain.csv", index=False)

## 4. Relation extraction by relation type (test)

In [ ]:
rt = S.relation_types(runs); display(rt)
if not rt.empty:
    display(Image(filename=str(S.plot_bars(rt, "relation", ["precision", "recall", "F1"], "Relation extraction per type (test)", f"{FIG}/relation_types.png")), width=640))

## 5. Evidence finder vs word-overlap search (QASPER test, ranking every passage of each paper)

In [ ]:
ev = S.evidence_ranking(runs); display(ev)
if not ev.empty:
    display(Image(filename=str(S.plot_bars(ev, "metric", list(ev.columns[1:]), "Finding the evidence paragraph (QASPER test)", f"{FIG}/evidence_ranking.png")), width=640))

## 6. Detailed reports (confusion matrices, per-class scores, significance tests)

In [ ]:
from researchlens.evaluation import reports
from pathlib import Path
for tag, fn in [("n1_ner", reports.ner_report), ("v2_figcls", reports.figcls_report)]:
    if Path(RES, tag).exists():
        print("=" * 30, tag)
        try:
            display(fn(RES))
        except Exception as e:
            print("report failed:", type(e).__name__, e)

## 7. Save all figures (download from the Output panel)

In [ ]:
!cd {ROOT} && zip -qr rl_figures.zip rl_figures && ls -la rl_figures.zip